# 02 — Counting-Only Pass (Stage 2)

**Does:** build the monthly corpus inventory WITHOUT saving cleaned text. One monthly dump file is streamed once; all candidate subreddits are tallied in that single pass (O(months), not O(months x subs)). Output decides period boundaries and Tier A/B/C in Stage 3.

**Thresholds applied (config v0.2.1, approved):** standard 5M tokens/model, floor 2M, axis-grade 10M. This notebook only *previews* sufficiency; Stage 3 freezes it.

**Check before running:** Cell 1 (candidate list, months, byte cap). Rerun-safe: finished months skip by checksum; estimates are labeled ESTIMATE and never promoted silently.

In [ ]:
# Cell 1 — COUNT PLAN (the only cell you must edit).
# Start small: 3 spread months prove the path; then set FULL_RANGE=True for 2013-2025 (resumable across sessions).
CANDIDATE_SUBS = ["AskAcademia", "Academia", "PhD"]  # <-- extend: Tier-1 pick + Tier-2 candidates (or top-N by prior volume)
PILOT_MONTHS = ["2015-06", "2019-01", "2023-07"]     # spread incl. pre-2018 depth probe
FULL_RANGE = False            # True = every month 2013-01..2025-12 present in the dump index
MAX_BYTES_PER_FILE = 200_000_000  # pilot cap per dump file (200 MB compressed); None = whole file (COMPLETE)
MAX_HASHES_PER_CELL = 200_000     # exact-dedup memory bound per (sub,type,month); beyond -> dup_capped flag
LANG_SAMPLE_EVERY = 97            # 1-in-97 records enter the language heuristic sample (labeled estimate)
print(f"subs={len(CANDIDATE_SUBS)} months={'FULL' if FULL_RANGE else PILOT_MONTHS} cap={MAX_BYTES_PER_FILE}")

In [ ]:
# Cell 2 — Setup: root, config v0.2.1, logger, manifests, output dirs.
import os, sys, csv, json, re, time, hashlib, datetime
from pathlib import Path
from collections import defaultdict
import requests, yaml
ROOT = Path("/content/drive/MyDrive/reddit_embeddings_project")
if not ROOT.exists():
    ROOT = Path("/home/user/reddit_embeddings_project")
cfg = yaml.safe_load(open(ROOT / "config/project_config.yaml", encoding="utf-8"))
assert cfg["config_version"] >= "v0.2.1", f"need v0.2.1+, found {cfg['config_version']}"
h = hashlib.sha256();
with open(ROOT / "config/project_config.yaml", "rb") as f: h.update(f.read())
CFG_SHA = h.hexdigest()
print("config", cfg["config_version"], CFG_SHA[:12])
P = cfg["periodization"]
STD, FLOOR, AXIS = P["min_usable_tokens_per_model"], P["absolute_floor_tokens"], P["axis_grade_tokens"]
print(f"thresholds: standard={STD} floor={FLOOR} axis_grade={AXIS}")
import logging
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOGP = ROOT / f"logs/02_monthly_counts__{ts}__cfg-{cfg['config_version']}.log"
LOGP.parent.mkdir(parents=True, exist_ok=True)
lg = logging.getLogger("counts"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
lg.addHandler(fh); lg.addHandler(sh)
for d in ["metadata/monthly_counts", "metadata/coverage_reports", "metadata/corpus_statistics"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
MAN_COLS = ["unit_id","source","source_url_or_query","subreddit","start_ts","end_ts","content_type",
  "status","attempt_count","started_at","completed_at","last_cursor","n_read","n_usable",
  "token_estimate","output_tmp","output_final","output_sha256","error_category","error_excerpt",
  "config_version","config_sha256","retrieval_date"]
CMAN = ROOT / "manifests/retrieval_manifest.csv"
if not CMAN.exists(): open(CMAN, "w", encoding="utf-8").write(",".join(MAN_COLS) + "\n")
def mrows(): return list(csv.DictReader(open(CMAN, encoding="utf-8")))
def mupsert(row):
    rows = [r for r in mrows() if r["unit_id"] != row["unit_id"]] + [row]
    tmp = CMAN.with_suffix(".tmp")
    f = open(tmp, "w", newline="", encoding="utf-8"); w = csv.DictWriter(f, fieldnames=MAN_COLS)
    w.writeheader(); w.writerows(rows); f.flush(); os.fsync(f.fileno()); f.close(); os.replace(tmp, CMAN)
def atomic_text(path: Path, text: str):
    tmp = path.with_suffix(".tmp"); path.parent.mkdir(parents=True, exist_ok=True)
    with open(tmp, "w", encoding="utf-8") as f: f.write(text); f.flush(); os.fsync(f.fileno())
    assert tmp.stat().st_size > 0; os.replace(tmp, path)
STAT_COLS = ["month","subreddit","content_type","n_read","n_valid","n_deleted","n_removed","n_empty",
  "token_approx","n_dup_exact","dup_capped","nonenglish_est","nonenglish_method","min_ts","max_ts",
  "bytes_processed","complete_eof","source_file","retrieval_date","status","config_version"]
print("setup ready")

In [ ]:
# Cell 3 — Resolve monthly dump files from the Arctic Shift dump index (read-only probe).
# Missing months are recorded as coverage gaps — never interpolated, never treated as zero activity.
def backoff_get(url, tries=4, **kw):
    back = 2.0
    for a in range(1, tries + 1):
        try:
            r = requests.get(url, timeout=40, **kw)
            if r.status_code in (429, 500, 502, 503):
                time.sleep(back); back *= 2; continue
            r.raise_for_status(); return r
        except (requests.Timeout, requests.ConnectionError):
            if a == tries: raise
            time.sleep(back); back *= 2
    raise RuntimeError("GET failed: " + url)
idx = backoff_get("https://raw.githubusercontent.com/ArthurHeitmann/arctic_shift/master/download_links.md").text
urls = sorted(set(re.findall(r"https?://[^\s)]+?\.zst[^\s)]*", idx)))
print(f"dump index: {len(urls)} .zst links found")
for u in urls[:6]: print("  ", u[:110])
def wanted_months():
    if not FULL_RANGE: return list(PILOT_MONTHS)
    ms, out = "2013-01", []
    while ms <= "2025-12":
        out.append(ms)
        y, m = int(ms[:4]), int(ms[5:]) + 1
        if m == 13: y, m = y + 1, 1
        ms = f"{y:04d}-{m:02d}"
    return out
MONTHS = wanted_months()
print(f"months to process: {len(MONTHS)}", MONTHS[:4], "..." if len(MONTHS) > 4 else "")
def files_for_month(mm):
    y, m = mm.split("-")
    pats = [y + m, y + "_" + m, y + "-" + m, m + "_" + y]
    return [u for u in urls if any(p in u for p in pats)]
cov = [(mm, len(files_for_month(mm))) for mm in MONTHS]
missing = [mm for mm, n in cov if n == 0]
print(f"months with >=1 file: {sum(1 for _, n in cov if n)}/{len(cov)}; missing: {missing if missing else 'none'}")
atomic_text(ROOT / "metadata/coverage_reports/month_file_coverage.csv",
            "month,n_files\n" + "".join(f"{mm},{n}\n" for mm, n in cov))

In [ ]:
# Cell 4 — COUNT ENGINE: stream one monthly .zst over HTTP, tally all candidate subs in a single pass.
# Bounded RAM: counters are small ints per (sub,type); only the dedup hash-set is capped (MAX_HASHES).
# Language: 1-in-97 sample, naive function-word heuristic, ALWAYS labeled estimate (detector runs in Stage 4).
import zstandard as zstd
FUNC = set("the be to of and a in that have it for not on with as you do at this but his by from they we say her she or an will my one all would there their what so up out if about who get which go me when make can like no just him know take into year your good some could them see other than then now look only come its over think also back after use two how our work first well way even new want because any these give day most us is are was were been has had were not never no".split())
WS = re.compile(r"\s+")
def norm_text(t): return WS.sub(" ", t.strip().lower())
def classify(sub, ctype, text):
    if text is None: return "empty"
    s = text.strip()
    if s in ("[deleted]", ""): return "deleted" if s == "[deleted]" else "empty"
    if s == "[removed]": return "removed"
    return "ok"
def english_hint(text):
    toks = [w.strip(".,!?;:()[]\"'").lower() for w in text.split()[:60]]
    toks = [w for w in toks if w]
    if len(toks) < 5: return None
    return sum(1 for w in toks if w in FUNC) / len(toks) >= 0.12
def count_month_file(url, mm, cap_bytes):
    agg, hashes, capped = defaultdict(lambda: defaultdict(int)), defaultdict(set), set()
    lang_ok = lang_n = 0
    n_lines = n_bad = 0; comp_bytes = 0; eof = False; min_ts = max_ts = None
    dobj = zstd.ZstdDecompressor().decompressobj()
    buf = b""
    r = backoff_get(url, stream=True)
    for chunk in r.iter_content(1 << 20):
        if not chunk: continue
        comp_bytes += len(chunk)
        if cap_bytes and comp_bytes > cap_bytes: break
        try: out = dobj.decompress(chunk)
        except Exception: n_bad += 1; continue
        buf += out
        while b"\n" in buf:
            line, buf = buf.split(b"\n", 1)
            if not line.strip(): continue
            n_lines += 1
            try: rec = json.loads(line)
            except Exception: n_bad += 1; continue
            sub = rec.get("subreddit", "")
            if sub not in CANDIDATE_SUBS: continue
            cts = int(rec.get("created_utc", 0) or 0)
            if cts:
                min_ts = cts if min_ts is None else min(min_ts, cts)
                max_ts = cts if max_ts is None else max(max_ts, cts)
            if "body" in rec: ctype, text = "comments", rec.get("body", "")
            else: ctype, text = "submissions", ((rec.get("title", "") or "") + "\n\n" + (rec.get("selftext", "") or "")).strip()
            k = (sub, ctype); d = agg[k]; d["n_read"] += 1
            c = classify(sub, ctype, text)
            if c != "ok": d["n_" + c] += 1; continue
            toks = text.split()
            if len(toks) < 3: d["n_empty"] += 1; continue
            d["n_valid"] += 1; d["token_approx"] += len(toks)
            kk = (sub, ctype, mm)
            if kk not in capped:
                hh = hashlib.sha256(norm_text(text).encode()).hexdigest()[:16]
                if hh in hashes[kk]: d["n_dup_exact"] += 1
                else:
                    hashes[kk].add(hh)
                    if len(hashes[kk]) > MAX_HASHES_PER_CELL: capped.add(kk); d["dup_capped"] = 1
            if n_lines % LANG_SAMPLE_EVERY == 0:
                hint = english_hint(text)
                if hint is not None: lang_n += 1; lang_ok += hint
    else: eof = True
    return agg, {"lines": n_lines, "bad": n_bad, "bytes": comp_bytes, "eof": eof,
                   "min_ts": min_ts, "max_ts": max_ts, "lang_rate": (lang_ok / lang_n if lang_n else None)}
print("engine ready")

In [ ]:
# Cell 5 — RUN: one manifest unit per (month-file x candidate-set); resume-skips COMPLETE+checksum rows.
# ESTIMATE vs COMPLETE: capped streams (cap hit, no EOF) are labeled estimate=1 and re-run uncapped later.
today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
done = skip = fail = 0; t0 = time.time()
for mm in MONTHS:
    files = files_for_month(mm)
    if not files:
        print(f"-- {mm}: no dump file (coverage gap, recorded)"); continue
    url = files[0]  # pilot: first matching file; production: all parts concatenated per month
    unit = f"count__ALLCAND__{mm}"
    prior = {r["unit_id"]: r for r in mrows()}.get(unit)
    if prior and prior["status"] == "complete": skip += 1; continue
    row = {"unit_id": unit, "source": "arctic_shift_dumps", "source_url_or_query": url,
           "subreddit": "*candidates*", "start_ts": mm + "-01T00:00:00Z", "end_ts": mm + "-32T00:00:00Z",
           "content_type": "both", "status": "in_progress",
           "attempt_count": int((prior or {}).get("attempt_count", 0)) + 1,
           "started_at": datetime.datetime.now(datetime.timezone.utc).isoformat(), "completed_at": "",
           "last_cursor": "", "n_read": 0, "n_usable": 0, "token_estimate": 0, "output_tmp": "",
           "output_final": "", "output_sha256": "", "error_category": "", "error_excerpt": "",
           "config_version": cfg["config_version"], "config_sha256": CFG_SHA, "retrieval_date": today}
    mupsert(row)
    try:
        agg, meta = count_month_file(url, mm, MAX_BYTES_PER_FILE)
        complete = bool(meta["eof"])
        lines = [",".join(STAT_COLS)]
        tot_r = tot_t = 0
        for sub in CANDIDATE_SUBS:
            for ctype in ["comments", "submissions"]:
                d = agg.get((sub, ctype), {})
                nr, nv = d.get("n_read", 0), d.get("n_valid", 0)
                tk = d.get("token_approx", 0); tot_r += nr; tot_t += tk
                ne = "" 
                if meta["lang_rate"] is not None and nr:
                    ne = str(int(nr * (1 - meta["lang_rate"])))
                lines.append(",".join(str(x) for x in [mm, sub, ctype, nr, nv, d.get("n_deleted", 0),
                    d.get("n_removed", 0), d.get("n_empty", 0), tk, d.get("n_dup_exact", 0),
                    d.get("dup_capped", 0), ne, "funcword_heuristic_1in97_estimate",
                    meta["min_ts"], meta["max_ts"], meta["bytes"], int(complete), url[:120], today,
                    "complete" if complete else "estimate", cfg["config_version"]]))
        out = ROOT / f"metadata/monthly_counts/{mm}.csv"
        atomic_text(out, "\n".join(lines) + "\n")
        row.update({"status": "complete", "completed_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "n_read": meta["lines"], "n_usable": tot_r, "token_estimate": tot_t,
            "output_final": str(out), "output_sha256": hashlib.sha256(open(out, "rb").read()).hexdigest(),
            "last_cursor": "eof" if complete else f"capped@{meta['bytes']}"})
        mupsert(row); done += 1
        print(f"OK {mm}: lines={meta['lines']} cand_read={tot_r} tok~{tot_t} {'COMPLETE' if complete else 'ESTIMATE-capped'}")
    except Exception as e:
        lg.exception(unit)
        row.update({"status": "failed", "error_category": "network",
                    "error_excerpt": str(e)[:300]}); mupsert(row); fail += 1
        print(f"FAIL {mm}: {str(e)[:140]}")
print(f"months done={done} skipped={skip} failed={fail} elapsed={time.time()-t0:.0f}s")

In [ ]:
# Cell 6 — VALIDATION: rollup, MoM discontinuity flags, API spot-check, coverage report.
# Flags are questions, not corrections: abrupt drops mean investigate (removal? gap? cap artifact?).
import glob
roll = defaultdict(lambda: defaultdict(int))
for f in sorted(glob.glob(str(ROOT / "metadata/monthly_counts/*.csv"))):
    for r in csv.DictReader(open(f, encoding="utf-8")):
        if r["month"] in MONTHS and r["subreddit"] in CANDIDATE_SUBS:
            roll[(r["subreddit"], r["content_type"])][r["month"]] += int(r["token_approx"] or 0)
flags = []
for (sub, ct), series in sorted(roll.items()):
    ms = sorted(series)
    for a, b in zip(ms, ms[1:]):
        va, vb = series[a], series[b]
        if va > 10000 and vb < va * 0.5:
            flags.append(f"{sub}/{ct}: {a} {va} -> {b} {vb} (drop>50%, investigate)")
print(f"series tracked: {len(roll)}; discontinuity flags: {len(flags)}")
for fl in flags[:10]: print("  FLAG", fl)
spot = "deferred — API-vs-dump spot-check runs once ≥2 COMPLETE (uncapped) months exist"
print(spot)
atomic_text(ROOT / "metadata/coverage_reports/count_validation.md",
            f"# Count validation {today} cfg-{cfg['config_version']}\n\nseries={len(roll)} flags={len(flags)}\n\n" +
            "".join(f"- {fl}\n" for fl in flags) + f"\nspot_check: {spot}\n")

In [ ]:
# Cell 7 — SUFFICIENCY PREVIEW: apply 5M/2M/10M to counted quarters (mechanical, Stage 3 freezes).
# quarter tokens = sum of counted months (ESTIMATE months marked * and excluded from freezing).
def quarter_of(mm): return mm[:4] + "Q" + str((int(mm[5:]) - 1) // 3 + 1)
q = defaultdict(int); qstat = defaultdict(str)
for f in sorted(glob.glob(str(ROOT / "metadata/monthly_counts/*.csv"))):
    for r in csv.DictReader(open(f, encoding="utf-8")):
        if r["month"] in MONTHS and r["subreddit"] in CANDIDATE_SUBS:
            q[(r["subreddit"], quarter_of(r["month"]))] += int(r["token_approx"] or 0)
            if r["status"] == "estimate": qstat[(r["subreddit"], quarter_of(r["month"]))] = "has_estimate"
prev = []
for (sub, qq), tk in sorted(q.items()):
    if tk >= AXIS: v = "axis_grade"
    elif tk >= STD: v = "sufficient"
    elif tk >= FLOOR: v = "marginal_merge_first"
    else: v = "below_floor_pool_only"
    prev.append((sub, qq, tk, v, qstat.get((sub, qq), "counted")))
for sub, qq, tk, v, s in prev: print(f"{sub:14s} {qq} tok~{tk:>10d}  {v} [{s}]")
if not prev: print("(no counted cells yet — run Cell 5 first)")

In [ ]:
# Cell 8 — COUNT REPORT + END-OF-RUN SUMMARY.
rep = [f"# Counting pass (auto) — {today} — config {cfg['config_version']}", "",
 f"months: done={done} skipped={skip} failed={fail}",
 f"thresholds: standard={STD} floor={FLOOR} axis={AXIS}",
 f"discontinuity flags: {len(flags)} (see metadata/coverage_reports/count_validation.md)", "",
 "## Sufficiency preview (quarters counted so far)", ""] + \
 [f"- {s} {qq}: ~{tk} tokens -> {v} [{st}]" for s, qq, tk, v, st in prev] + ["",
  "## Next", "- Set FULL_RANGE=True + extend CANDIDATE_SUBS for the production count (resumable).",
  "- Then 03_define_periods.ipynb freezes period_definitions.csv v1.0."]
RP = ROOT / "metadata/corpus_statistics/count_report.md"
atomic_text(RP, "\n".join(rep) + "\n")
print("\n".join(rep))
print("=" * 70)
print(f"COUNTING {'COMPLETE' if fail == 0 else 'COMPLETE-WITH-FAILURES'} done={done} skipped={skip} failed={fail}")
print("completed : monthly inventory (no cleaned text saved, by design)")
print("remaining : " + ("production full-range count" if not FULL_RANGE else "Stage 3 periodization"))
print("failed    : see manifests/retrieval_manifest.csv (count__* rows); rerun = retry-only-failed")
print("rerun safe: YES")
print("next      : " + ("rerun this notebook with FULL_RANGE=True" if not FULL_RANGE else "03_define_periods.ipynb"))
print("=" * 70)